# B - Training strategy & hyperparameters (Cluster 4)

**Reviewer comments addressed**

* **R1.4 / R3.7** - no rationale for learning rate, epochs, batch size, or the
  number of collocation / boundary points; no study of their effect on accuracy
  and training time.

The paper uses **22,500** collocation points - this is `n_collocation = 150`
because the sampler builds a `150 x 150` tensor grid (`150**2 = 22500`). We
sweep the collocation count (and boundary count) and report accuracy + time, so
the choice is *justified empirically* rather than asserted.

In [ ]:
# === Environment setup (robust: local / Colab native / VSCode-Colab) ===
# Run this cell FIRST. It is idempotent (safe to re-run) and does NOT touch
# Colab's preinstalled numpy/scipy/torch (avoids ABI breakage) - it just puts
# the repo's `src/` on the path and installs the one missing dep, scikit-fem.
import os, sys, subprocess, importlib
REPO_URL = 'https://github.com/Daniel14gonc/PINNs_piezoelectricity.git'

def _find_repo():
    """Return the repo root (folder with src/pinn_piezo) at/above cwd."""
    d = os.getcwd()
    for _ in range(10):
        if os.path.isdir(os.path.join(d, 'src', 'pinn_piezo')):
            return d
        nd = os.path.dirname(d)
        if nd == d:
            return None
        d = nd
    return None

repo = _find_repo()
if repo is None:
    # Fresh runtime: clone ONCE into a fixed absolute path (no nesting on re-run).
    base = '/content' if os.path.isdir('/content') else os.getcwd()
    repo = os.path.join(base, 'PINNs_piezoelectricity')
    if not os.path.isdir(os.path.join(repo, 'src', 'pinn_piezo')):
        subprocess.run(['git', 'clone', REPO_URL, repo], check=True)
    else:
        subprocess.run(['git', '-C', repo, 'pull', '--ff-only'], check=False)

os.chdir(repo)
src = os.path.join(repo, 'src')
if src not in sys.path:
    sys.path.insert(0, src)   # import from source; NO pip install of the stack

# Only scikit-fem is missing on a stock Colab; install it (its deps already exist).
try:
    import skfem  # noqa: F401
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'scikit-fem'], check=True)

# Verify the revision modules made it into the checkout (i.e. you pushed them).
missing = [m for m in ('pinn_piezo', 'pinn_piezo.fem', 'pinn_piezo.metrics',
                       'pinn_piezo.indirect.standard')
           if importlib.util.find_spec(m) is None]
assert not missing, ('Missing modules: ' + ', '.join(missing) +
                     ' - push them to GitHub, then `git -C ' + repo +
                     ' pull` (or restart runtime) and re-run.')

import torch, pinn_piezo
print('pinn_piezo :', pinn_piezo.__file__)
print('repo       :', repo)
print('torch      :', torch.__version__, '| cuda:', torch.cuda.is_available())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Where this notebook writes its tables / figures.
from pathlib import Path
OUT = Path('outputs/revision'); OUT.mkdir(parents=True, exist_ok=True)
print('Artefacts ->', OUT.resolve())

In [ ]:
# The geometry/collocation datasets (data/*.npy) are gitignored, so a fresh
# clone does not have them. Regenerate the paper-size sets if missing
# (n_collocation=150 -> 150**2 = 22,500 collocation points).
import os
from pinn_piezo import geometry
for suffix in ('_m1', '_m1_d'):
    if not os.path.exists(f'data/xy_top_non_normalized{suffix}.npy'):
        geometry.generate_and_save(n_points=400, n_collocation=150,
                                   n_collocation_test=200, suffix=suffix,
                                   data_dir='data')
        print('generated data', suffix)
    else:
        print('data present', suffix)

In [ ]:
import numpy as np, pandas as pd, torch
torch.set_default_dtype(torch.float64)
from pinn_piezo import geometry, metrics, fem
from pinn_piezo.indirect import model as imodel, train as itrain
from pinn_piezo.indirect.train import tensorize

# Reference for scoring.
# Reference fields from the validated scikit-fem solver (Notebook 00 shows it
# matches the analytical solution; poling_sign=-1 = this repo's indirect convention).
r = fem.solve_piezo('indirect', nx=300, ny=10, voltage=100.0, poling_sign=-1.0)
df = pd.DataFrame({'X_Coordinate': r.points[:,0], 'Y_Coordinate': r.points[:,1],
                   'X_Deflection': r.u, 'Y_Deflection': r.v, 'Potential': r.phi})
XY = df[['X_Coordinate','Y_Coordinate']].values
REF = {'u': df['X_Deflection'].values, 'v': df['Y_Deflection'].values, 'phi': df['Potential'].values}

QUICK = True
import os
EP_ADAM  = int(os.environ.get('REV_ADAM',  200 if QUICK else 1000))
EP_LBFGS = int(os.environ.get('REV_LBFGS', 50  if QUICK else 200))

def train_and_score(n_collocation, n_points=400, seed=0):
    np.random.seed(seed); torch.manual_seed(seed)
    geometry.generate_and_save(n_points=n_points, n_collocation=n_collocation,
                               n_collocation_test=50, suffix='_sweep', data_dir='data')
    arrays = itrain.load_dataset('data', suffix='_sweep', fraction=1.0)
    tensors = itrain.to_device(arrays, DEVICE, dtype=torch.float64)
    model = imodel.build_default_model(device=DEVICE, model_type='pyramid')
    res = itrain.train(model, tensors, epochs_adam=EP_ADAM, epochs_lbfgs=EP_LBFGS)
    p = model(tensorize(XY, DEVICE)).detach().cpu().numpy()
    mt = metrics.metrics_table({'u':p[:,0],'v':p[:,1],'phi':p[:,2]}, REF)
    return {'n_collocation_grid': n_collocation, 'n_points_total': n_collocation**2,
            'time_s': res['total_time'], 'L2_u': mt.loc['u','rel_L2'],
            'L2_v': mt.loc['v','rel_L2'], 'L2_phi': mt.loc['phi','rel_L2']}

## 1. Collocation-point sweep (~5000 / 10000 / 22500 points)

In [ ]:
# 71**2≈5041, 100**2=10000, 150**2=22500 (the paper).
GRIDS = [71, 100, 150]
rows = [train_and_score(n) for n in GRIDS]
coll = pd.DataFrame(rows).set_index('n_collocation_grid')
coll.to_csv(OUT / 'sweep_collocation.csv')
coll

## 2. Boundary-point sweep

In [ ]:
def train_and_score_bc(n_points, n_collocation=150, seed=0):
    np.random.seed(seed); torch.manual_seed(seed)
    geometry.generate_and_save(n_points=n_points, n_collocation=n_collocation,
                               n_collocation_test=50, suffix='_bcsweep', data_dir='data')
    arrays = itrain.load_dataset('data', suffix='_bcsweep', fraction=1.0)
    tensors = itrain.to_device(arrays, DEVICE, dtype=torch.float64)
    model = imodel.build_default_model(device=DEVICE, model_type='pyramid')
    res = itrain.train(model, tensors, epochs_adam=EP_ADAM, epochs_lbfgs=EP_LBFGS)
    p = model(tensorize(XY, DEVICE)).detach().cpu().numpy()
    mt = metrics.metrics_table({'u':p[:,0],'v':p[:,1],'phi':p[:,2]}, REF)
    return {'n_points': n_points, 'n_boundary_pts_per_edge': arrays['xy_right'].shape[0],
            'time_s': res['total_time'], 'L2_u': mt.loc['u','rel_L2'],
            'L2_v': mt.loc['v','rel_L2'], 'L2_phi': mt.loc['phi','rel_L2']}

rows = [train_and_score_bc(n) for n in (200, 400, 600)]
bc = pd.DataFrame(rows).set_index('n_points')
bc.to_csv(OUT / 'sweep_boundary.csv'); bc

## 3. (Optional) learning-rate / epoch note

In [ ]:
# A small LR comparison to justify lr=1e-3 (Adam). Extend as needed.
import numpy as np
arrays = itrain.load_dataset('data', suffix='_m1', fraction=1.0)
tensors = itrain.to_device(arrays, DEVICE, dtype=torch.float64)
rows = []
for lr in (1e-2, 1e-3, 1e-4):
    np.random.seed(0); torch.manual_seed(0)
    model = imodel.build_default_model(device=DEVICE, model_type='pyramid')
    res = itrain.train(model, tensors, epochs_adam=EP_ADAM, epochs_lbfgs=0, lr_adam=lr)
    rows.append({'lr_adam': lr, 'final_loss': res['loss_list'][-1], 'time_s': res['total_time']})
pd.DataFrame(rows).set_index('lr_adam')

---
### Rebuttal snippet (Cluster 4)
> *Hyper-parameters were selected through preliminary experiments balancing
> convergence and computational cost. We now report the sensitivity of the
> accuracy and training time to the number of collocation points
> (Table …; 22,500 corresponds to the 150×150 sampling grid) and boundary
> points (Table …), and the effect of the learning rate (Table …). Accuracy
> saturates beyond … points, justifying the chosen values.*